In [1]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="https://hpi-s3.mittelstand-zukunftslab.de",
    aws_access_key_id="rustfsadmin",
    aws_secret_access_key="ohhxvube4hanww8rjjjj",
    region_name="us-east-1",
)

response = s3.list_buckets()
print([b["Name"] for b in response["Buckets"]])

['deepfix-lancedb']


In [2]:
import lancedb

db = lancedb.connect(
    "s3://deepfix-lancedb/legal_rag_bench",
    storage_options={
        "endpoint": "https://hpi-s3.mittelstand-zukunftslab.de",
        "aws_access_key_id": "rustfsadmin",
        "aws_secret_access_key": "ohhxvube4hanww8rjjjj",
        "aws_region": "us-east-1",
    },
)
print(db.table_names())

[]


C:\Users\agrim\AppData\Local\Temp\ipykernel_25516\767924398.py:12: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  print(db.table_names())


In [5]:
LITELLM_URL = "https://litellm.mittelstand-zukunftslab.de"
LITELLM_KEY = "sk-pfrgLhvL2Idn3HCWC0yPAQ"

In [17]:
from datasets import load_dataset
qa_dataset = load_dataset("isaacus/legal-rag-bench", "qa", split="test")
print(len(qa_dataset))

100


In [3]:
from datasets import load_dataset

corpus_dataset = load_dataset("isaacus/legal-rag-bench", "corpus", split="test")
print(len(corpus_dataset))

C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4876


In [6]:
import litellm
import pandas as pd

def embed_batch(texts, batch_size=10):
    all_vectors = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        resp = litellm.embedding(
            model="openai/qwen/qwen3-embedding-8b",
            input=chunk,
            api_key=LITELLM_KEY,
            api_base=LITELLM_URL,
            timeout=60
        )
        all_vectors.extend([d["embedding"] for d in resp.data])
    return all_vectors

test_rows = corpus_dataset.select(range(20))
test_texts = [r["text"] for r in test_rows]
test_vectors = embed_batch(test_texts)

test_data = [
    {"id": row["id"], "title": row["title"], "text": row["text"], "vector": vec}
    for row, vec in zip(test_rows, test_vectors)
]

test_df = pd.DataFrame(test_data)
table = db.create_table("legal_passages_test", data=test_df, mode="overwrite")
print(table.count_rows())

20


In [9]:
query_text = "jury excuse bias"
query_vector = embed_batch([query_text])[0]

results = (
    table.search(query_type="hybrid")
    .vector(query_vector)
    .text(query_text)
    .limit(3)
    .to_pandas()
)
print(results[["id", "title", "text"]])

            id                title  \
0  1.2.1-c3-s1  1.2.1 Other Reasons   
1    1.2-c2-s1  1.2 Excusing Jurors   
2    1.2-c2-s3  1.2 Excusing Jurors   

                                                text  
0  ## Other Reasons\n\nFinally, there may also be...  
1                                 ## Excusing Jurors  
2  7. There is no absolute right of an accused pe...  


In [10]:
full_texts = [r["text"] for r in corpus_dataset]
print(f"Embedding {len(full_texts)} passages... this will take a while")

full_vectors = embed_batch(full_texts, batch_size=10)
print(f"Done embedding: {len(full_vectors)} vectors")

Embedding 4876 passages... this will take a while
Done embedding: 4876 vectors


In [11]:
full_data = [
    {"id": row["id"], "title": row["title"], "text": row["text"], "vector": vec}
    for row, vec in zip(corpus_dataset, full_vectors)
]

full_df = pd.DataFrame(full_data)
full_table = db.create_table("legal_passages", data=full_df, mode="overwrite")
print(full_table.count_rows())

4876


In [12]:
from lancedb.index import FTS
full_table.create_index("text", config=FTS())

In [23]:
def hybrid_answer_question_v2(question, top_k=10):
    q_vector = embed_batch([question])[0]
    results = (
        full_table.search(query_type="hybrid")
        .vector(q_vector)
        .text(question)
        .limit(top_k)
        .to_pandas()
    )
    context_chunks = list(results["text"])
    context_str = "\n\n---\n\n".join(context_chunks)

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so.

Context:
{context_str}

Question: {question}

Answer:"""

    chat_response = litellm.completion(
        model="openai/openrouter/qwen/qwen3.6-plus",
        messages=[{"role": "user", "content": prompt}],
        api_key=LITELLM_KEY,
        api_base=LITELLM_URL,
        timeout=60
    )
    return {
        "context": context_chunks,
        "answer": chat_response.choices[0].message.content
    }

In [19]:
def diagnose_lancedb(qa_index, limit=20):
    qa = qa_dataset[qa_index]
    question = qa["question"]
    correct_id = qa["relevant_passage_id"]

    fts_results = full_table.search(question, query_type="fts").limit(limit).to_pandas()
    fts_ids = list(fts_results["id"])

    q_vector = embed_batch([question])[0]
    vec_results = full_table.search(q_vector).limit(limit).to_pandas()
    vec_ids = list(vec_results["id"])

    print(f"Q: {question[:70]}...")
    print(f"Correct ID: {correct_id}")
    print(f"FTS-only top {limit}: found={correct_id in fts_ids} | ids={fts_ids}")
    print(f"Vector-only top {limit}: found={correct_id in vec_ids} | ids={vec_ids}")
    print("---")

for i in [11, 28, 21]:
    diagnose_lancedb(i)

Q: Does Victoria have a law/laws protecting against double jeopardy?...
Correct ID: 3.12-c1-s1
FTS-only top 20: found=False | ids=['8.14-c8-s1', '8.14.3-c11-s1', '8.14.3-c3-s1', '8.15-c2-s2', '8.9-c12-s1', '8.10-c12-s1', '8.14.2-c9-s1', '7.5.16-c6-s1', '8.1-c9-s1', '7.7.2.1-c3-s1', '8.14.3-c4-s1', '2.5.1-c1-s1', '8.14.2-c5-s2', '8.14.3-c7-s2', '8.14.3-c2-s1', '8.14.3-c7-s1', '8.14.2-c5-s1', '8.14.2-c3-s1', '7.4.12-c4-s1', '7.4.13-c4-s1']
Vector-only top 20: found=False | ids=['7.7.1-c14-s1', '5.9-c1-s1', '4.11-c6-s4', '1.2-c2-s1', '2.2-c4-s11', '7.4.12-c7-s1', '7.3.23-c8-s1', '4.1-c6-s4', '8.2-c4-s1', '7.3.12.1-c3-s1', '8.3-c2-s1', '1.7-c4-s1', '7.8.3-c2-s7', '3.10-c4-s1', '4.13.4-c4-s1', '8.5-c3-s1', '5.4-c4-s1', '3.10-c3-s1', '3.13-c3-s4', '7.6.1-c9-s2']
---
Q: Chris is on trial for intentionally or recklessly causing a bushfire. ...
Correct ID: 4.13.1-c2-s1
FTS-only top 20: found=False | ids=['7.5.21.1-c1-s1', '7.5.21.2-c1-s1', '7.5.21.1-c4-s1', '4.13.1-c2-s2', '7.5.21-c1-s1', '7.7.

In [20]:
def test_hybrid_wider(qa_index, limit=10):
    qa = qa_dataset[qa_index]
    question = qa["question"]
    correct_id = qa["relevant_passage_id"]
    q_vector = embed_batch([question])[0]

    results = (
        full_table.search(query_type="hybrid")
        .vector(q_vector)
        .text(question)
        .limit(limit)
        .to_pandas()
    )
    ids = list(results["id"])
    print(f"Q: {question[:70]}...")
    print(f"Correct ID: {correct_id} | Found: {correct_id in ids} | Rank: {ids.index(correct_id)+1 if correct_id in ids else 'N/A'}")
    print(f"All retrieved: {ids}")
    print("---")

for i in [11, 28, 21]:
    test_hybrid_wider(i, limit=10)

Q: Does Victoria have a law/laws protecting against double jeopardy?...
Correct ID: 3.12-c1-s1 | Found: False | Rank: N/A
All retrieved: ['7.7.1-c14-s1', '8.14-c8-s1', '5.9-c1-s1', '8.14.3-c11-s1', '4.11-c6-s4', '8.14.3-c3-s1', '7.4.12-c7-s1', '8.15-c2-s2', '1.2-c2-s1', '8.9-c12-s1']
---
Q: Chris is on trial for intentionally or recklessly causing a bushfire. ...
Correct ID: 4.13.1-c2-s1 | Found: False | Rank: N/A
All retrieved: ['7.5.21.1-c4-s1', '7.5.21.2-c1-s1', '7.5.21.1-c1-s1', '7.5.21-c1-s1', '7.5.21-c2-s1', '4.13.1-c2-s2', '7.5.19.1-c5-s1', '7.7.1.4-c4-s3', '7.5.21.1-c3-s1', '5.5-c2-s9']
---
Q: The defence raises the issue of mental impairment and bears the onus o...
Correct ID: 4.9-c2-s1 | Found: True | Rank: 7
All retrieved: ['8.4-c4-s1', '8.4.1-c2-s1', '10.2.1-c6-s2', '8.4-c2-s1', '8.4.2-c1-s1', '10.1-c4-s1', '4.9-c2-s1', '8.8-c4-s10', '8.4.1-c3-s1', '8.4.1-c1-s1']
---


In [24]:
from deepeval.test_case import LLMTestCase
import time

test_cases_v2 = []
failed_v2 = []

for qa in qa_dataset:
    try:
        r = hybrid_answer_question_v2(qa["question"])
        test_cases_v2.append(LLMTestCase(
            input=qa["question"],
            actual_output=r["answer"],
            expected_output=qa["answer"],
            retrieval_context=r["context"],
        ))
    except Exception as e:
        failed_v2.append({"id": qa.get("id"), "error": str(e)})
    time.sleep(0.5)

print(f"Succeeded: {len(test_cases_v2)} / {len(qa_dataset)}")
print(f"Failed: {len(failed_v2)}")

Succeeded: 100 / 100
Failed: 0


In [25]:
import nest_asyncio
nest_asyncio.apply()

from deepeval.models.base_model import DeepEvalBaseLLM

class ProxyJudgeLLM(DeepEvalBaseLLM):
    def __init__(self, model, api_base, api_key):
        self.model, self.api_base, self.api_key = model, api_base, api_key
    def load_model(self):
        return self.model
    def generate(self, prompt: str) -> str:
        r = litellm.completion(model=self.model, messages=[{"role":"user","content":prompt}],
                                api_key=self.api_key, api_base=self.api_base, timeout=60)
        return r.choices[0].message.content
    async def a_generate(self, prompt: str) -> str:
        r = await litellm.acompletion(model=self.model, messages=[{"role":"user","content":prompt}],
                                       api_key=self.api_key, api_base=self.api_base, timeout=60)
        return r.choices[0].message.content
    def get_model_name(self):
        return self.model

judge = ProxyJudgeLLM("openai/openrouter/openai/gpt-4.1-mini", LITELLM_URL, LITELLM_KEY)
print(judge.generate("Say OK"))

OK


In [26]:
from deepeval.metrics import FaithfulnessMetric, ContextualPrecisionMetric, GEval
from deepeval.test_case import LLMTestCaseParams

faithfulness = FaithfulnessMetric(threshold=0.80, model=judge, async_mode=False)
contextual_precision = ContextualPrecisionMetric(threshold=0.75, model=judge, async_mode=False)
answer_correctness = GEval(
    name="Answer Correctness",
    criteria="Determine whether the actual output is factually correct and complete relative to the expected output, without omitting key legal conclusions.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    threshold=0.75,
    model=judge,
    async_mode=False,
)

C:\Users\agrim\AppData\Local\Temp\ipykernel_25516\1504915439.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [27]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig

eval_results_v2 = evaluate(
    test_cases_v2,
    metrics=[faithfulness, contextual_precision, answer_correctness],
    async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Faithfulness Metric! (using openai/openrouter/openai/gpt-4.1-mini, 
strict=False, async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openai/openrouter/openai/gpt-4.1-mini, 
strict=False, async_mode=False)...

✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using 
openai/openrouter/openai/gpt-4.1-mini, strict=False, async_mode=False)...

C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

ValueError: Evaluation LLM outputted an invalid JSON. Please use a better evaluation model.

In [28]:
import pandas as pd

metrics_list = [("Faithfulness", faithfulness), ("Contextual Precision", contextual_precision), ("Answer Correctness", answer_correctness)]

results_v2 = []
errors_v2 = []

for idx, tc in enumerate(test_cases_v2):
    row = {"input": tc.input, "actual_output": tc.actual_output}
    for name, metric in metrics_list:
        success = False
        for attempt in range(3):
            try:
                metric.measure(tc)
                row[f"{name}_score"] = metric.score
                row[f"{name}_reason"] = metric.reason
                success = True
                break
            except Exception as e:
                last_error = str(e)
        if not success:
            row[f"{name}_score"] = None
            row[f"{name}_reason"] = f"FAILED after 3 attempts: {last_error}"
            errors_v2.append({"index": idx, "metric": name, "error": last_error})

    results_v2.append(row)

    if idx % 5 == 0:
        pd.DataFrame(results_v2).to_csv("deepeval_results_lancedb_v2_partial.csv", index=False)
        print(f"Checkpoint saved at question {idx+1}/{len(test_cases_v2)}")

pd.DataFrame(results_v2).to_csv("deepeval_results_lancedb_v2_final.csv", index=False)
print(f"Done. Total errors: {len(errors_v2)}")

C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Checkpoint saved at question 1/100


C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Checkpoint saved at question 6/100


C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Checkpoint saved at question 91/100


C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Checkpoint saved at question 96/100


Done. Total errors: 0


In [29]:
df_v2 = pd.DataFrame(results_v2)

summary_v2 = df_v2[["Faithfulness_score", "Contextual Precision_score", "Answer Correctness_score"]].mean()
print(summary_v2)

print("\nPer-metric pass rate (score >= threshold):")
print(f"Faithfulness >= 0.80: {(df_v2['Faithfulness_score'] >= 0.80).mean():.2%}")
print(f"Contextual Precision >= 0.75: {(df_v2['Contextual Precision_score'] >= 0.75).mean():.2%}")
print(f"Answer Correctness >= 0.75: {(df_v2['Answer Correctness_score'] >= 0.75).mean():.2%}")

Faithfulness_score            0.893526
Contextual Precision_score    0.654646
Answer Correctness_score      0.732000
dtype: float64

Per-metric pass rate (score >= threshold):
Faithfulness >= 0.80: 78.00%
Contextual Precision >= 0.75: 42.00%
Answer Correctness >= 0.75: 71.00%


In [30]:
worst_precision = df_v2.sort_values("Contextual Precision_score").head(5)
for _, r in worst_precision.iterrows():
    print(r["input"])
    print("Score:", r["Contextual Precision_score"], "| Reason:", r["Contextual Precision_reason"])
    print("---")

Harry is serving as a juror in a burglary trial. He is also a professional locksmith. During the proceedings, Harry is shown a lock which, in his expert view, is too damaged to have been lockpicked in the manner described in court. Why might Harry's expert knowledge of lockpicking be irrelevant when assessing the physical evidence?
Score: 0.0 | Reason: The score is 0.00 because all the nodes in retrieval contexts are irrelevant, with 'no' verdicts at every rank. For example, the first node discusses general guidelines on expert evidence but does not address the irrelevance of expert knowledge due to changes in physical evidence, showing that irrelevant nodes are ranked higher than any relevant ones. Similarly, other nodes ranked from second to tenth also fail to touch on the expert knowledge relevance given the condition of the evidence, explaining why the precision score is at zero.
---
Does Victoria have a law/laws protecting against double jeopardy?
Score: 0.0 | Reason: The score is

In [31]:
check_ids = [
    "Harry is serving as a juror",
    "Does Victoria have a law",
    "One of the elements of aggravated carjacking",
    "At dinner, Kim and Frank",
    "Two friends agree to race"
]
for c in check_ids:
    match = df_v2[df_v2["input"].str.contains(c[:20], regex=False)]
    print(match[["input", "Faithfulness_score", "Answer Correctness_score"]].to_string())
    print("---")

                                                                                                                                                                                                                                                                                                                                           input  Faithfulness_score  Answer Correctness_score
1  Harry is serving as a juror in a burglary trial. He is also a professional locksmith. During the proceedings, Harry is shown a lock which, in his expert view, is too damaged to have been lockpicked in the manner described in court. Why might Harry's expert knowledge of lockpicking be irrelevant when assessing the physical evidence?            0.333333                       0.3
---
                                                                input  Faithfulness_score  Answer Correctness_score
11  Does Victoria have a law/laws protecting against double jeopardy?                 1.0                       0.

In [32]:
low_prec = df_v2[df_v2["Contextual Precision_score"] < 0.2]
high_prec = df_v2[df_v2["Contextual Precision_score"] >= 0.75]

print(f"Zero/near-zero precision (n={len(low_prec)}):")
print(low_prec[["Faithfulness_score", "Answer Correctness_score"]].mean())
print()
print(f"High precision (n={len(high_prec)}):")
print(high_prec[["Faithfulness_score", "Answer Correctness_score"]].mean())

Zero/near-zero precision (n=11):
Faithfulness_score          0.878788
Answer Correctness_score    0.290909
dtype: float64

High precision (n=42):
Faithfulness_score          0.885317
Answer Correctness_score    0.838095
dtype: float64


In [42]:
def rewrite_query_for_retrieval(question, retries=3, timeout=60):
    prompt = f"""Read this legal question. In one short phrase, name the specific legal doctrine, rule, or concept it is testing (not the case facts).

Question: {question}

Legal concept:"""
    for attempt in range(retries):
        try:
            r = litellm.completion(model="openai/openrouter/qwen/qwen3.6-plus",
                                    messages=[{"role":"user","content":prompt}],
                                    api_key=LITELLM_KEY, api_base=LITELLM_URL, timeout=timeout)
            return r.choices[0].message.content.strip()
        except Exception as e:
            if attempt == retries - 1:
                return f"FAILED: {e}"
            time.sleep(3)

In [43]:
failing_questions = low_prec["input"].tolist()
rewritten_concepts = []
for q in failing_questions:
    rewritten = rewrite_query_for_retrieval(q)
    rewritten_concepts.append(rewritten)
    print("ORIGINAL:", q[:80])
    print("REWRITTEN CONCEPT:", rewritten)
    print("---")

ORIGINAL: Harry is serving as a juror in a burglary trial. He is also a professional locks
REWRITTEN CONCEPT: Rule against juror expertise
---
ORIGINAL: Does Victoria have a law/laws protecting against double jeopardy?
REWRITTEN CONCEPT: Double jeopardy
---
ORIGINAL: One of the elements of aggravated carjacking is the use of a weapon or causing i
REWRITTEN CONCEPT: Weapon focus effect
---
ORIGINAL: At dinner, Kim and Frank watch a news story about a drive-by shooting. Kim says 
REWRITTEN CONCEPT: Adoptive admission
---
ORIGINAL: A person is sitting in the back seat of a moving taxi when the taxi strikes a pe
REWRITTEN CONCEPT: Constructive driving
---
ORIGINAL: Two friends agree to race their cars along a suburban street. One driver loses c
REWRITTEN CONCEPT: Joint enterprise
---
ORIGINAL: The prosecution alleges that the accused forced the complainant to put his own p
REWRITTEN CONCEPT: Statutory definition of sexual penetration
---
ORIGINAL: Assault' is defined to mean the direct or 

In [44]:
def test_rewritten_retrieval(original_q, rewritten_q):
    q_vector = embed_batch([rewritten_q])[0]
    results = (
        full_table.search(query_type="hybrid")
        .vector(q_vector)
        .text(rewritten_q)
        .limit(10)
        .to_pandas()
    )
    print("ORIGINAL:", original_q[:80])
    print("REWRITTEN:", rewritten_q)
    print("TOP 3 RETRIEVED CHUNKS:")
    for t in results["text"].head(3):
        print(" -", t[:150])
    print("---")

for q, r in zip(failing_questions, rewritten_concepts):
    test_rewritten_retrieval(q, r)

ORIGINAL: Harry is serving as a juror in a burglary trial. He is also a professional locks
REWRITTEN: Rule against juror expertise
TOP 3 RETRIEVED CHUNKS:
 - ## Excusing Jurors
 - Third, acting on outside information would be false to the oath or affirmation you took as jurors to give a true verdict according to the evidence. Yo
 - ## Other Directions Concerning Jury Deliberations
---
ORIGINAL: Does Victoria have a law/laws protecting against double jeopardy?
REWRITTEN: Double jeopardy
TOP 3 RETRIEVED CHUNKS:
 - ## Duplicity and Multiple Offences
 - (a) children can accurately remember and report past events; and

(b) children are developing language and cognitive skills, and this may affect–

(i)
 - ## Onus of Proof
---
ORIGINAL: One of the elements of aggravated carjacking is the use of a weapon or causing i
REWRITTEN: Weapon focus effect
TOP 3 RETRIEVED CHUNKS:
 - ## Intention
 - [*If it is alleged that the relevant article was made for the use of injuring or incapacitating a person

In [45]:
def hybrid_answer_question_v3(question, top_k=10):
    concept = rewrite_query_for_retrieval(question)
    combined_query = f"{question}\n\nRelevant legal concept: {concept}"

    q_vector = embed_batch([combined_query])[0]
    results = (
        full_table.search(query_type="hybrid")
        .vector(q_vector)
        .text(combined_query)
        .limit(top_k)
        .to_pandas()
    )
    context_chunks = list(results["text"])
    context_str = "\n\n---\n\n".join(context_chunks)

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so.

Context:
{context_str}

Question: {question}

Answer:"""

    chat_response = litellm.completion(
        model="openai/openrouter/qwen/qwen3.6-plus",
        messages=[{"role": "user", "content": prompt}],
        api_key=LITELLM_KEY,
        api_base=LITELLM_URL,
        timeout=60
    )
    return {"context": context_chunks, "answer": chat_response.choices[0].message.content}

In [46]:
failing_ids = low_prec.index.tolist()
test_cases_v3 = []

for idx in failing_ids:
    qa = qa_dataset[idx]
    r = hybrid_answer_question_v3(qa["question"])
    test_cases_v3.append(LLMTestCase(
        input=qa["question"], actual_output=r["answer"],
        expected_output=qa["answer"], retrieval_context=r["context"]
    ))

results_v3 = []
for tc in test_cases_v3:
    row = {"input": tc.input}
    for name, metric in metrics_list:
        try:
            metric.measure(tc)
            row[f"{name}_score"] = metric.score
        except Exception as e:
            row[f"{name}_score"] = None
    results_v3.append(row)

df_v3 = pd.DataFrame(results_v3)
print(df_v3.mean(numeric_only=True))


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Faithfulness_score            1.000000
Contextual Precision_score    0.264571
Answer Correctness_score      0.663636
dtype: float64


In [47]:
import pandas as pd

df_v2_final = pd.DataFrame(results_v2)

comparison = pd.DataFrame({
    "system": ["Qdrant_baseline", "LanceDB_hybrid_v2"],
    "faithfulness": [0.877, df_v2_final["Faithfulness_score"].mean()],
    "contextual_precision": [0.71, df_v2_final["Contextual Precision_score"].mean()],
    "answer_correctness": [0.593, df_v2_final["Answer Correctness_score"].mean()],
    "n_questions": [100, 100]
})

df_v2_final.to_csv("deepeval_lancedb_v2_per_question.csv", index=False)
comparison.to_csv("deepeval_lancedb_vs_qdrant_summary.csv", index=False)
print(comparison)

              system  faithfulness  contextual_precision  answer_correctness  \
0    Qdrant_baseline      0.877000              0.710000               0.593   
1  LanceDB_hybrid_v2      0.893526              0.654646               0.732   

   n_questions  
0          100  
1          100  
